<a href="https://colab.research.google.com/github/bps-rajora/Misinformation-detector/blob/main/NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install pandas scikit-learn nltk scipy

In [6]:
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from scipy.sparse import hstack

In [7]:
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"

df = pd.read_table(url, header=None, names=['label','text'])

df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [8]:
df['label'] = df['label'].map({'ham':0, 'spam':1})

df.head()

,label,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [9]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z ]", "", text)
    return text

df['clean_text'] = df['text'].apply(clean_text)

In [10]:
def count_caps(text):
    return sum(1 for c in text if c.isupper())

def count_links(text):
    return text.count("http")

def count_exclamations(text):
    return text.count("!")

In [11]:
df['caps'] = df['text'].apply(count_caps)
df['links'] = df['text'].apply(count_links)
df['exclam'] = df['text'].apply(count_exclamations)

df.head()

,label,text,clean_text,caps,links,exclam
0,0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...,3,0,0
1,0,Ok lar... Joking wif u oni...,ok lar joking wif u oni,2,0,0
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in a wkly comp to win fa cup final...,10,0,0
3,0,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say,2,0,0
4,0,"Nah I don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...,2,0,0


In [12]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=3000)

X_text = vectorizer.fit_transform(df['clean_text'])

In [13]:
X_extra = df[['caps','links','exclam']].values

X = hstack([X_text, X_extra])

y = df['label']

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [15]:
model = LogisticRegression()

model.fit(X_train, y_train)

LogisticRegression()

In [16]:
pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

Accuracy: 0.968609865470852


In [17]:
def predict_message(text):

    clean = clean_text(text)
    vec = vectorizer.transform([clean])

    caps = count_caps(text)
    links = count_links(text)
    exclam = count_exclamations(text)

    extra = np.array([[caps,links,exclam]])

    final = hstack([vec, extra])

    pred = model.predict(final)

    if pred[0] == 1:
        return "Spam / Possible misinformation"
    else:
        return "Normal message"

In [18]:
predict_message("WIN FREE BITCOIN NOW!!! CLICK HERE!!!")

'Spam / Possible misinformation'

In [27]:
predict_message("you won a car")

'Normal message'

In [24]:
predict_message("URGENT! You've won a FREE iPhone! Claim now at this link: http://fakeurl.com")

'Spam / Possible misinformation'

Let's test the model with another message:

In [19]:
predict_message("Hey, are we still meeting for lunch today?")

'Normal message'